In [ ]:
import cv2
import cv2.aruco as aruco
import numpy as np

# --- CONFIGURATION ---
VERTICAL_OFFSET = -1.5   # How far "up" the vector to look (1.0 = 1 tag height)
BOX_SCALE = 0.8         # Size of the target box

# Calibrated Colors
COLOR_RANGES = {
    "RED": [ (np.array([170, 80, 40]), np.array([180, 255, 255])),
             (np.array([0, 80, 40]), np.array([10, 255, 255])) ],
    "BLUE": [ (np.array([100, 150, 40]), np.array([140, 255, 255])) ]
}

def get_vector_box(corner_array, frame_shape):
    # CRITICAL FIX: Reshape the array to (4, 2) to ensure safe unpacking
    # corner_array comes in as (1, 4, 2), we flatten it to (4, 2)
    pts = corner_array.reshape(4, 2)
    
    # Unpack corners: TopLeft, TopRight, BottomRight, BottomLeft
    (tl, tr, br, bl) = pts

    # 1. Calculate the "Up" Vector (Left edge and Right edge)
    # This vector points from Bottom -> Top
    vec_left = tl - bl
    vec_right = tr - br
    vec_avg = (vec_left + vec_right) / 2.0
    
    # 2. Calculate Center of Tag
    cx = int(np.mean(pts[:, 0]))
    cy = int(np.mean(pts[:, 1]))
    
    # 3. Project Target Center
    # Start at Tag Center -> Move along "Up Vector" * Offset
    target = np.array([cx, cy]) + (vec_avg * VERTICAL_OFFSET)
    tx, ty = int(target[0]), int(target[1])
    
    # 4. Calculate Size (based on physical tag pixel height)
    tag_h = np.linalg.norm(vec_avg)
    box_size = int(tag_h * BOX_SCALE)
    
    # Calculate Top-Left of the target box
    sx = int(tx - (box_size / 2))
    sy = int(ty - (box_size / 2))
    
    return sx, sy, box_size, (cx, cy), (tx, ty)

def identify_color(img):
    if img is None or img.size == 0: return "Unknown", (200, 200, 200)
    
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    tot = img.shape[0] * img.shape[1]
    
    found_name = "Scanning"
    draw_color = (0, 0, 0)
    max_c = 0
    
    for name, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype="uint8")
        for l, u in ranges:
            mask = cv2.bitwise_or(mask, cv2.inRange(hsv, l, u))
            
        c = cv2.countNonZero(mask)
        if c > (tot * 0.05) and c > max_c:
            max_c = c
            found_name = name
            if name == "RED": draw_color = (0, 0, 255)
            elif name == "BLUE": draw_color = (255, 0, 0)
            
    return found_name, draw_color

def safe_crop(frame, x, y, size):
    h, w = frame.shape[:2]
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(w, x + size), min(h, y + size)
    if x1 >= x2 or y1 >= y2: return None
    return frame[y1:y2, x1:x2]

def run_stable_detector():
    cap = cv2.VideoCapture(0)
    
    dictionary = aruco.getPredefinedDictionary(aruco.DICT_APRILTAG_36h11)
    parameters = aruco.DetectorParameters()
    detector = aruco.ArucoDetector(dictionary, parameters)

    print("System Active. Press 'q' to quit.")

    try:
        while True:
            ret, frame = cap.read()
            if not ret: break

            corners, ids, rejected = detector.detectMarkers(frame)

            if ids is not None:
                aruco.drawDetectedMarkers(frame, corners, ids)
                
                for i in range(len(ids)):
                    # Get Geometry
                    sx, sy, size, center_pt, target_pt = get_vector_box(corners[i], frame.shape)
                    
                    # Identify
                    roi = safe_crop(frame, sx, sy, size)
                    status, color = identify_color(roi)
                    
                    # Draw visual "Vector" line (Yellow)
                    cv2.line(frame, center_pt, target_pt, (0, 255, 255), 2)
                    
                    # Draw Box
                    cv2.rectangle(frame, (sx, sy), (sx+size, sy+size), color, 2)
                    cv2.putText(frame, status, (sx, sy-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            cv2.imshow('FRC Hub Vision', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
            
    except Exception as e:
        print(f"Error: {e}")
    finally:
        cap.release()
        cv2.destroyAllWindows()

run_stable_detector()